# Capstone RAG — Conversational Memory (LCEL)

This notebook extends the base RAG pipeline with **conversational memory**: the
assistant remembers earlier turns, so follow-up questions with pronouns
("what about *that*?", "who wrote *it*?") are understood in context.

How it works:
1. **Contextualize** — an LLM step rewrites the follow-up into a *standalone*
   question using the chat history, so retrieval gets a self-contained query.
2. **Retrieve** — the standalone question hits the Chroma retriever.
3. **Answer** — the LLM answers using the retrieved context **and** the chat
   history, with your exact `ChatPromptTemplate` instructions.

The chain is **stateless**: history is passed in on each call. The caller
(here the notebook, in the app `st.session_state`) owns the conversation.

In [ ]:
import sys
from pathlib import Path

# make notebooks/src importable (pdf_ingestion.py lives there)
sys.path.append(str(Path.cwd() / "src"))
sys.path.append(str(Path.cwd().parent / "src"))

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(Path.cwd().parent / ".env")
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))

## 1. Ingest the PDF into chunks

In [ ]:
from pdf_ingestion import PdfIngestion

DATA_DIR = Path.cwd().parent / "data" / "pdf_files"
pdf_path = next(DATA_DIR.glob("*.pdf"))

ingestor = PdfIngestion(chunk_size=1200, chunk_overlap=180)
pdf_chunks = ingestor.process(str(pdf_path))
print(f"Total chunks: {len(pdf_chunks)} from {pdf_path.name}")

## 2. Build the vector store and retriever

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embed = OpenAIEmbeddings(model="text-embedding-3-small")

# In-memory (no persist_directory): small corpus, fast to embed at startup,
# and avoids ephemeral-filesystem issues on hosted platforms (same as the app).
pdf_vector_store = Chroma.from_documents(
    documents=pdf_chunks,
    embedding=embed,
    collection_name="capstone_focused_docs",
)

retriever = pdf_vector_store.as_retriever(search_kwargs={"k": 3})

## 3. Prompts

Two prompts:
- **`contextualize_prompt`** — turns a follow-up + history into a standalone question.
- **`answer_prompt`** — your original `ChatPromptTemplate` wording, made
  message-based so it can also carry the chat history.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

contextualize_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given the chat history and the latest user question, which might "
            "reference earlier turns, rephrase it into a standalone question "
            "understandable without the chat history. Do NOT answer it — only "
            "reformulate if needed, otherwise return it unchanged.",
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{question}"),
    ]
)

# Your exact ChatPromptTemplate wording, now message-based (+ chat history).
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a technical assistant for our data analytics team.\n"
            "Answer the question below focusing on the context below.\n"
            'If there is no answer in the context, just say: "there is no answer"\n\n'
            "CONTEXT:\n{context}\n\n"
            "Be precise and very concise.",
        ),
        MessagesPlaceholder("chat_history"),
        ("human", "{question}"),
    ]
)

## 4. Build the conversational chain (LCEL)

In [ ]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        f"[page {d.metadata.get('page', '?')}] {d.page_content}" for d in docs
    )


# Reformulate only when there is history to resolve (saves an LLM call on turn 1).
contextualize = contextualize_prompt | llm | parser


def standalone_question(x: dict) -> str:
    if x.get("chat_history"):
        return contextualize.invoke(x)
    return x["question"]


retrieve_context = RunnableLambda(standalone_question) | retriever | format_docs

conversational_rag = (
    RunnablePassthrough.assign(context=retrieve_context)
    | answer_prompt
    | llm
    | parser
)

## 5. Talk to it — memory across turns

`chat_history` is a list of LangChain messages covering the turns **before**
the current question. `chat()` below keeps that list and appends each turn,
exactly like the Streamlit app does with `st.session_state`.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

chat_history = []  # grows as the conversation proceeds


def chat(question: str) -> str:
    answer = conversational_rag.invoke(
        {"question": question, "chat_history": chat_history}
    )
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))
    print(f"You: {question}\nAssistant: {answer}\n")
    return answer


_ = chat("What is this report about?")
_ = chat("What are its main findings about that decline?")  # 'its' / 'that' need memory
_ = chat("And what does it recommend?")

## Notes

- This is the same logic packaged in [`src/rag.py`](src/rag.py)
  (`build_conversational_rag`) and served by the Streamlit app (`app.py`).
- The app converts its `st.session_state.messages` into LangChain messages with
  `to_lc_messages(...)` and passes them as `chat_history` on every turn — the
  chain itself stays stateless and cache-safe.